In [18]:
import operator
from typing import Annotated, TypedDict


class AgentState(TypedDict):
    messages: Annotated[list[dict], operator.add]


from langchain.tools import tool
from langgraph.types import interrupt


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送邮件工具

    - 在发送之前会中断， 等待用户统一

    Args:
        to (str): 收件人
        subject (str): 邮件主题
        body (str): 邮件内容

    Returns:
        str: 发送结果
    """
    print("send_email")
    response = interrupt({
        "action": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "message": "是否发送该邮件",
    })
    if response.get("action") == "approve":
        final_to = response.get("to", to)
        final_subject = response.get("subject", subject)
        final_body = response.get("body", body)
        print(f"[Send Email] to ={final_to}, subject={final_subject}, body={final_body}")
        return f"邮件已发送至 {final_to}"
    return "已经取消邮件发送"


import os

import dotenv
from langchain.chat_models import init_chat_model

dotenv.load_dotenv()

llm = init_chat_model(
    model="deepseek-chat",
    base_url="https://api.deepseek.com/v1",
    api_key=os.getenv("DEEPSEEK_API"),
    model_provider="openai",
    temperature=0.0,
).bind_tools([send_email])

from langchain.messages import ToolMessage


def agent_node(state: AgentState) -> AgentState:
    result = llm.invoke(state["messages"])
    return {"messages": [result]}


def tool_node(state: AgentState) -> AgentState:
    print(state)
    last_msg = state["messages"][-1]
    tool_calls = getattr(last_msg, "tool_calls", [])
    tool_messages = []
    for tool_call in tool_calls:
        print(tool_call)
        result = send_email.invoke(tool_call["args"])
        print(result)
        tool_messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
    return {"message": tool_messages}


from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tool", tool_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", "tool")
builder.add_edge("tool", END)
grpah = builder.compile(InMemorySaver())

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "test"}}
state_0 = grpah.invoke(
    {
        "messages": [
            HumanMessage(content="给邮箱地址为 123456@qq.com 的用户发邮件;邮件主题为: 你好; 邮件的内容为: 我是agent")
        ]
    },
    config=config,
)
print(state_0)
print(state_0.get("__interrupt__", {}))

from langgraph.types import Command

resumed = grpah.invoke(
    Command(
        resume={
            "action": "approve",
            "subject": "会议通知",
            "body": "20202",
        }
    ),
    config=config,
)

print(resumed)


{'messages': [HumanMessage(content='给邮箱地址为 123456@qq.com 的用户发邮件;邮件主题为: 你好; 邮件的内容为: 我是agent', additional_kwargs={}, response_metadata={}), AIMessage(content='好的，我来发送这封邮件。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 375, 'total_tokens': 463, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 119}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '56bac70a-efdf-4fc5-8c89-d90003c0e3fb', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f3ac3-3786-7f12-84e2-590794534bbd-0', tool_calls=[{'name': 'send_email', 'args': {'to': '123456@qq.com', 'subject': '你好', 'body': '我是agent'}, 'id': 'call_00_29naV8GdehWo60LR7Kp58175', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 375, 'outpu